# Acehnese → Indonesian: train our model and publish the free web app
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Menu **Runtime → Run all**.
3. The first cell asks for your **Hugging Face WRITE token** (huggingface.co/settings/tokens). Paste it, press **Enter**.
4. Wait about 40–60 minutes with this tab open. The last cell prints the link to your app.

If something goes wrong, the notebook keeps going: without a finished training run it still publishes the app with the
base NLLB model, so you always get a working link. Copy any red error text and send it to Claude.

In [ ]:
# ---- settings ----------------------------------------------------------------------------------
from getpass import getpass
HF_TOKEN = getpass('Paste your Hugging Face WRITE token, then press Enter (Enter alone = do not publish): ').strip()
MODEL_PRIVATE = True   # True: the model files stay private, the app is still public. Set False once Andrie confirms the licences.
USE_QURAN = False      # extra training data (Acehnese Quran translation): only after Andrie agrees
EPOCHS = 10            # maximum; training stops earlier when it stops improving

In [ ]:
# ---- code + packages + GPU check ---------------------------------------------------------------
import os
REPO = os.path.abspath('aceh_code')
if not os.path.isdir(REPO):
    !git clone -q -b claude/dataset-product-build-108xu8 https://github.com/Mutaib-yye/aceh_code {REPO}
%cd {REPO}
!git pull -q
!pip install -q "transformers==5.19.0" "sentencepiece>=0.2" pymupdf pysbd datasketch rapidfuzz sacrebleu
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Menu: Runtime > Change runtime type > T4 GPU > Save. Then Runtime > Run all again.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ---- training data: NusaX + FLORES-200 + Hikayat Abu Sammah -> data/final/combined (leak-checked) ----
!python -m acehid fetch-nusax
if not os.path.exists('data/final/flores/dev.json'):
    !wget -q -O /tmp/flores200_dataset.tar.gz https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz && python -m acehid flores /tmp/flores200_dataset.tar.gz || echo 'FLORES download failed: training without it (still fine)'
extra = '--quran' if USE_QURAN else ''
if USE_QURAN:
    !python -m acehid fetch-quran
!python -m acehid build-train {extra}
import json
m = json.load(open('data/final/combined/manifest.json'))
print('training pairs:', m['train']['n'], m['train']['by_source'])
print('validation:', m['validation']['n'], '| test sets:', {k: v['n'] for k, v in m['tests'].items()})

In [ ]:
# ---- fine-tune NLLB-200 (about 20-40 minutes on a T4) ----------------------------------------------
!python -m acehid train --epochs {EPOCHS}
if not os.path.exists('models/ace-id-nllb/RESULTS.md'):
    print('First attempt did not finish (usually: not enough GPU memory). Retrying with a smaller batch...')
    !python -m acehid train --epochs {EPOCHS} --batch-size 4
TRAINED = os.path.exists('models/ace-id-nllb/RESULTS.md')
print('TRAINING FINISHED' if TRAINED else 'TRAINING FAILED: copy the red error above and send it to Claude. The app will be published with the base model.')

In [ ]:
# ---- results: base model vs ours on test sentences it never saw ----------------------------------
from IPython.display import Markdown, display
if TRAINED:
    display(Markdown(open('models/ace-id-nllb/RESULTS.md', encoding='utf-8').read()))
else:
    print('No results table: training did not finish.')

In [ ]:
# ---- try it (change the sentences if you like) ----------------------------------------------------
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from acehid.normalize import normalize
MODEL_DIR = 'models/ace-id-nllb' if TRAINED else 'facebook/nllb-200-distilled-600M'
tok = AutoTokenizer.from_pretrained(MODEL_DIR, src_lang='ace_Latn')
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to('cuda').eval()
def tr(text):
    x = tok([normalize(text)[0]], return_tensors='pt').to('cuda')
    with torch.no_grad():
        y = model.generate(**x, forced_bos_token_id=tok.convert_tokens_to_ids('ind_Latn'), num_beams=4, max_new_tokens=128)
    return tok.batch_decode(y, skip_special_tokens=True)[0]
for s in ['Lon jak u peukan uroe nyoe.', 'Teuma troih le Abu Sammah, budak indah paraih rupa']:
    print(s, '->', tr(s))

In [ ]:
# ---- publish: model (private by default) + translator app (public) on Hugging Face ----------------
import shutil
if not HF_TOKEN:
    print('No token given, so nothing was published. Run the first cell again with your token, then this cell.')
else:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    try:
        user = api.whoami()['name']
    except Exception as e:
        raise RuntimeError('Hugging Face did not accept the token. Create a WRITE token at huggingface.co/settings/tokens and run the first cell again.') from e
    space_repo = f'{user}/penerjemah-aceh'
    if TRAINED:
        model_repo = f'{user}/ace-id-nllb'
        fp16 = 'models/ace-id-nllb-fp16'               # half precision: 1.2 GB instead of 2.4 GB, same translations
        model.half().save_pretrained(fp16); tok.save_pretrained(fp16)
        for f in ('training_log.json', 'RESULTS.md', 'README.md'):
            shutil.copy(f'models/ace-id-nllb/{f}', fp16)
        api.create_repo(model_repo, private=MODEL_PRIVATE, exist_ok=True)
        api.update_repo_settings(model_repo, private=MODEL_PRIVATE)
        api.upload_folder(repo_id=model_repo, folder_path=fp16, commit_message='Fine-tuned Acehnese -> Indonesian NLLB')
    else:
        model_repo = 'facebook/nllb-200-distilled-600M'
    api.create_repo(space_repo, repo_type='space', space_sdk='gradio', private=False, exist_ok=True)
    api.add_space_variable(space_repo, 'MODEL_ID', model_repo)
    if TRAINED and MODEL_PRIVATE:
        api.add_space_secret(space_repo, 'HF_TOKEN', HF_TOKEN)   # lets the public app load your private model
    api.upload_folder(repo_id=space_repo, repo_type='space', folder_path='space', commit_message='Translator app',
                      ignore_patterns=['__pycache__/*', '*.pyc'])
    print('APP  :', f'https://huggingface.co/spaces/{space_repo}', '(first start takes about 5-10 minutes)')
    print('MODEL:', f'https://huggingface.co/{model_repo}', '(private)' if TRAINED and MODEL_PRIVATE else '')